In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv("dataset/IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


## Model Architecture: LSTM for Sentiment Classification

We will implement a **many-to-one LSTM architecture** from scratch using NumPy for binary sentiment classification on the IMDb dataset.

### Architecture

1. **Input:** Movie reviews from the IMDb dataset.
2. **Tokenisation:** Convert reviews into sequences of integer token IDs.
3. **Padding and Truncation:** Standardise each review to a maximum length of 100 tokens.
4. **Embedding Layer:** Trainable embedding matrix that converts each token ID into a 32-dimensional vector.
5. **LSTM Layer:** A single LSTM layer with 64 hidden units, processing the input sequence sequentially.
6. **Dense Layer:** Maps the final hidden state (h_T) to 2 output logits.
7. **Softmax:** Converts logits into probabilities for the two sentiment classes:
   - 0 — Negative
   - 1 — Positive

### Initial Hyperparameters

| Parameter | Value |
|---|---:|
| Vocabulary size | 10,000 |
| Embedding dimension | 32 |
| Maximum sequence length | 100 |
| Number of LSTM layers | 1 |
| Hidden dimension | 64 |
| Output classes | 2 |
| Batch size | 32 |

### Implementation Approach

- NumPy-based implementation from scratch.
- Manual forward propagation and Backpropagation Through Time (BPTT).
- Trainable word embeddings and LSTM parameters.
- Model selection based on validation loss.
- Save the best model checkpoint, including learned parameters and configuration, for future reuse.

**Architecture flow:**

`Review → Tokenisation → Embedding → LSTM → Dense → Softmax → Sentiment Prediction`

In [4]:
print(df.shape)
print(df.columns)

(50000, 2)
Index(['review', 'sentiment'], dtype='str')


In [5]:
df.sentiment=df.sentiment.map({"negative":0,"positive":1})
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. <br /><br />The...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1


In [7]:
df["review"] = df["review"].str.lower()

In [9]:
df["tokens"] = df["review"].apply(lambda x: x.split())

In [10]:
print("Original review:\n", df["review"].iloc[0][:300])
print("\nTokenised review:\n", df["tokens"].iloc[0][:30])
print("\nNumber of tokens:", len(df["tokens"].iloc[0]))

Original review:
 one of the other reviewers has mentioned that after watching just 1 oz episode you'll be hooked. they are right, as this is exactly what happened with me.<br /><br />the first thing that struck me about oz was its brutality and unflinching scenes of violence, which set in right from the word go. tru

Tokenised review:
 ['one', 'of', 'the', 'other', 'reviewers', 'has', 'mentioned', 'that', 'after', 'watching', 'just', '1', 'oz', 'episode', "you'll", 'be', 'hooked.', 'they', 'are', 'right,', 'as', 'this', 'is', 'exactly', 'what', 'happened', 'with', 'me.<br', '/><br', '/>the']

Number of tokens: 307


In [6]:
from sklearn.model_selection import train_test_split

In [12]:
df.head()

,review,sentiment,tokens
0,one of the other reviewers has mentioned that ...,1,"[one, of, the, other, reviewers, has, mentione..."
1,a wonderful little production. <br /><br />the...,1,"[a, wonderful, little, production., <br, /><br..."
2,i thought this was a wonderful way to spend ti...,1,"[i, thought, this, was, a, wonderful, way, to,..."
3,basically there's a family where a little boy ...,0,"[basically, there's, a, family, where, a, litt..."
4,"petter mattei's ""love in the time of money"" is...",1,"[petter, mattei's, ""love, in, the, time, of, m..."


In [11]:
X = df["tokens"]
y = df["sentiment"]

In [13]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y,test_size=0.30,random_state=42,stratify=y)

In [14]:
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp,test_size=0.50,random_state=42,stratify=y_temp)

In [16]:
print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Testing samples:", len(X_test))

Training samples: 35000
Validation samples: 7500
Testing samples: 7500


In [17]:
word_counts = {}

for review in X_train:
    for word in review:
        if word in word_counts:
            word_counts[word] += 1
        else:
            word_counts[word] = 1

print("Unique words in training data:", len(word_counts))

Unique words in training data: 311486


In [18]:
vocab_size = 10000

sorted_words = sorted(word_counts.items(),key=lambda x: x[1],reverse=True)

word_to_id = {"<PAD>": 0,"<OOV>": 1}

for idx, (word, count) in enumerate(sorted_words[:vocab_size - 2], start=2):
    word_to_id[word] = idx

print("Final vocabulary size:", len(word_to_id))

Final vocabulary size: 10000


In [24]:
sorted_words[:5]

[('the', 445582),
 ('a', 221116),
 ('and', 218872),
 ('of', 199975),
 ('to', 184684)]

In [25]:
word_to_id

{'<PAD>': 0,
 '<OOV>': 1,
 'the': 2,
 'a': 3,
 'and': 4,
 'of': 5,
 'to': 6,
 'is': 7,
 'in': 8,
 'i': 9,
 'this': 10,
 'that': 11,
 'it': 12,
 '/><br': 13,
 'was': 14,
 'as': 15,
 'for': 16,
 'with': 17,
 'but': 18,
 'on': 19,
 'movie': 20,
 'are': 21,
 'his': 22,
 'not': 23,
 'you': 24,
 'film': 25,
 'have': 26,
 'be': 27,
 'he': 28,
 'one': 29,
 'at': 30,
 'by': 31,
 'an': 32,
 'they': 33,
 'from': 34,
 'all': 35,
 'who': 36,
 'like': 37,
 'so': 38,
 'just': 39,
 'or': 40,
 'has': 41,
 'about': 42,
 'her': 43,
 "it's": 44,
 'if': 45,
 'some': 46,
 'out': 47,
 'what': 48,
 'when': 49,
 'very': 50,
 'there': 51,
 'more': 52,
 'would': 53,
 'even': 54,
 'good': 55,
 'she': 56,
 'my': 57,
 'their': 58,
 'no': 59,
 'only': 60,
 'really': 61,
 'had': 62,
 'which': 63,
 'up': 64,
 'can': 65,
 'see': 66,
 'were': 67,
 'than': 68,
 'we': 69,
 '-': 70,
 'been': 71,
 'get': 72,
 'will': 73,
 'into': 74,
 'much': 75,
 'because': 76,
 'story': 77,
 'most': 78,
 'how': 79,
 'other': 80,
 "don't":

In [26]:
def encode_review(review, word_to_id):
    return [word_to_id.get(word, 1) for word in review]

In [27]:
X_train_encoded = [encode_review(review, word_to_id) for review in X_train]
X_val_encoded = [encode_review(review, word_to_id) for review in X_val]
X_test_encoded = [encode_review(review, word_to_id) for review in X_test]

In [28]:
print("Original review:", X_train.iloc[0][:15])
print("Encoded review:", X_train_encoded[0][:15])

print("\nEncoded review length:", len(X_train_encoded[0]))

Original review: ['if', 'you', 'want', 'to', 'see', 'a', 'true', 'thriller,', 'rent', 'this!!!', "it's", 'not', 'from', 'the', 'director']
Encoded review: [45, 24, 172, 6, 66, 3, 311, 3407, 817, 1, 44, 23, 34, 2, 205]

Encoded review length: 53


In [37]:
max_length = 100
pad_id = 0

def pad_sequence(sequence, max_length):
    if len(sequence) < max_length:
        sequence = sequence + [pad_id] * (max_length - len(sequence))
    else:
        sequence = sequence[:max_length]

    return sequence


In [38]:
X_train_padded = np.array([
    pad_sequence(review, max_length) for review in X_train_encoded
])

X_val_padded = np.array([
    pad_sequence(review, max_length) for review in X_val_encoded
])

X_test_padded = np.array([
    pad_sequence(review, max_length) for review in X_test_encoded
])


In [40]:
print("Training shape:", X_train_padded.shape)
print("Validation shape:", X_val_padded.shape)
print("Testing shape:", X_test_padded.shape)

Training shape: (35000, 100)
Validation shape: (7500, 100)
Testing shape: (7500, 100)


In [59]:
np.random.seed(42)

vocab_size = 10000
embedding_size = 32

embedding_matrix = np.random.randn(vocab_size, embedding_size) * 0.1

# Keep padding embedding fixed at zero
embedding_matrix[0] = 0

In [62]:
X_train_embedded = embedding_matrix[X_train_padded]
X_val_embedded = embedding_matrix[X_val_padded]
X_test_embedded = embedding_matrix[X_test_padded]

In [63]:
input_dim = embedding_size
hidden_dim = 64
output_dim = 2

In [69]:
# Input gate
W_i = np.random.randn(hidden_dim, input_dim) * 0.1
U_i = np.random.randn(hidden_dim, hidden_dim) * 0.1
b_i = np.zeros(hidden_dim)

# Forget gate
W_f = np.random.randn(hidden_dim, input_dim) * 0.1
U_f = np.random.randn(hidden_dim, hidden_dim) * 0.1
b_f = np.zeros(hidden_dim)

# Output gate
W_o = np.random.randn(hidden_dim, input_dim) * 0.1
U_o = np.random.randn(hidden_dim, hidden_dim) * 0.1
b_o = np.zeros(hidden_dim)

# Candidate cell state
W_c = np.random.randn(hidden_dim, input_dim) * 0.1
U_c = np.random.randn(hidden_dim, hidden_dim) * 0.1
b_c = np.zeros(hidden_dim)

# output layer
W_y = np.random.randn(output_dim, hidden_dim) * 0.1
b_y = np.zeros(output_dim)

In [70]:
for name, param in {
    "W_i": W_i, "U_i": U_i, "b_i": b_i,
    "W_f": W_f, "U_f": U_f, "b_f": b_f,
    "W_o": W_o, "U_o": U_o, "b_o": b_o,
    "W_c": W_c, "U_c": U_c, "b_c": b_c,
    "W_y": W_y, "b_y": b_y
}.items():
    print(name, param.shape)

W_i (64, 32)
U_i (64, 64)
b_i (64,)
W_f (64, 32)
U_f (64, 64)
b_f (64,)
W_o (64, 32)
U_o (64, 64)
b_o (64,)
W_c (64, 32)
U_c (64, 64)
b_c (64,)
W_y (2, 64)
b_y (2,)


In [74]:
batch_size=32

In [75]:
h_0 = np.zeros((batch_size, hidden_dim))
c_0 = np.zeros((batch_size, hidden_dim))

In [76]:
## Defining the sigmoid function

In [77]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

In [84]:
X_train_embedded

array([[[ 6.62881269e-02,  1.17347386e-01,  1.81021559e-02, ...,
          4.44602668e-02,  1.19663149e-01, -6.09782904e-02],
        [-6.26967058e-02,  1.81244856e-01,  7.07751935e-02, ...,
          6.81500697e-02,  2.83183761e-03,  2.97561395e-03],
        [-4.83617140e-02,  8.41780775e-02, -1.05122003e-01, ...,
          2.10496299e-02, -1.60419382e-02,  2.50090050e-01],
        ...,
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00]],

       [[ 9.69959650e-03,  5.95157025e-02, -8.18220683e-02, ...,
         -2.13447152e-02,  3.10907566e-02,  1.47535622e-01],
        [-4.71931866e-02,  1.08895060e-01,  6.42800191e-03, ...,
          1.03246526e-01, -1.51936997e

In [125]:

def forward_pass(X_tokens_batch, parameters):

    batch_size = X_tokens_batch.shape[0]
    sequence_length = X_tokens_batch.shape[1]

    # Initialise hidden state and cell state
    h_t = np.zeros((batch_size, hidden_dim))
    c_t = np.zeros((batch_size, hidden_dim))

    # Store intermediate values for BPTT
    cache = []

    # Iterate through all time steps
    for t in range(sequence_length):

        # Current token IDs
        token_ids = X_tokens_batch[:, t]

        # Retrieve current word embeddings
        x_t = parameters["embedding_matrix"][token_ids]

        # Identify padding positions
        mask = (token_ids != 0).reshape(-1, 1)

        # Store previous states
        h_prev = h_t.copy()
        c_prev = c_t.copy()

        # -----------------------------
        # 1. Input Gate
        # -----------------------------
        i_t = sigmoid(
            x_t @ parameters["W_i"].T +
            h_prev @ parameters["U_i"].T +
            parameters["b_i"]
        )

        # -----------------------------
        # 2. Forget Gate
        # -----------------------------
        f_t = sigmoid(
            x_t @ parameters["W_f"].T +
            h_prev @ parameters["U_f"].T +
            parameters["b_f"]
        )

        # -----------------------------
        # 3. Candidate Cell State
        # -----------------------------
        c_tilde = np.tanh(
            x_t @ parameters["W_c"].T +
            h_prev @ parameters["U_c"].T +
            parameters["b_c"]
        )

        # -----------------------------
        # 4. Cell State Update
        # -----------------------------
        c_new = f_t * c_prev + i_t * c_tilde

        # -----------------------------
        # 5. Output Gate
        # -----------------------------
        o_t = sigmoid(
            x_t @ parameters["W_o"].T +
            h_prev @ parameters["U_o"].T +
            parameters["b_o"]
        )

        # -----------------------------
        # 6. Hidden State Update
        # -----------------------------
        h_new = o_t * np.tanh(c_new)

        # Preserve states at padding positions
        c_t = np.where(mask, c_new, c_prev)
        h_t = np.where(mask, h_new, h_prev)

        # -----------------------------
        # 7. Store Cache for BPTT
        # -----------------------------
        cache.append({
            "token_ids": token_ids.copy(),
            "x_t": x_t.copy(),
            "mask": mask.copy(),
            "h_prev": h_prev,
            "c_prev": c_prev,
            "i_t": i_t,
            "f_t": f_t,
            "o_t": o_t,
            "c_tilde": c_tilde,
            "c_t": c_t.copy(),
            "h_t": h_t.copy()
        })

    # -----------------------------
    # 8. Dense Output Layer
    # -----------------------------
    logits = h_t @ parameters["W_y"].T + parameters["b_y"]

    # -----------------------------
    # 9. Softmax
    # -----------------------------
    logits = logits - np.max(logits, axis=1, keepdims=True)

    exp_logits = np.exp(logits)

    probabilities = exp_logits / np.sum(
        exp_logits, axis=1, keepdims=True
    )

    return probabilities, cache, h_t


In [126]:
def cross_entropy_loss(probabilities, y_true):
    epsilon = 1e-12
    correct_probs = probabilities[np.arange(len(y_true)), y_true]
    loss = -np.mean(np.log(correct_probs + epsilon))
    return loss

In [127]:
y_batch = y_train.iloc[:batch_size].to_numpy()

In [128]:
loss = cross_entropy_loss(probabilities, y_batch)

print("Cross-Entropy Loss:", loss)

Cross-Entropy Loss: 0.694146689136033


In [145]:
def backward_pass(probabilities, y_true, h_final, cache, parameters):

    batch_size = len(y_true)

    # Initialise gradients
    grads = {
        name: np.zeros_like(param)
        for name, param in parameters.items()
    }

    # =====================================
    # 1. OUTPUT LAYER GRADIENTS
    # =====================================

    y_one_hot = np.eye(output_dim)[y_true]

    d_logits = (probabilities - y_one_hot) / batch_size

    grads["W_y"] = d_logits.T @ h_final
    grads["b_y"] = np.sum(d_logits, axis=0)

    # Gradient flowing into final hidden state
    dh_next = d_logits @ parameters["W_y"]

    # Gradient flowing into final cell state
    dc_next = np.zeros((batch_size, hidden_dim))

    # =====================================
    # 2. BACKPROPAGATION THROUGH TIME
    # =====================================

    for t in reversed(range(len(cache))):

        step = cache[t]

        # Retrieve cached values
        x_t = step["x_t"]
        token_ids = step["token_ids"]
        mask = step["mask"]

        h_prev = step["h_prev"]
        c_prev = step["c_prev"]

        i_t = step["i_t"]
        f_t = step["f_t"]
        o_t = step["o_t"]
        c_tilde = step["c_tilde"]

        c_t = step["c_t"]

        # =====================================
        # 3. HIDDEN STATE GRADIENT
        # =====================================

        dh_future = dh_next.copy()

        # Apply padding mask
        dh = dh_future * mask

        tanh_c = np.tanh(c_t)

        # Output gate gradient
        do = dh * tanh_c

        da_o = do * o_t * (1 - o_t)

        # Gradient flowing into cell state
        dc = dc_next + dh * o_t * (1 - tanh_c**2)

        # =====================================
        # 4. CELL STATE GRADIENT
        # =====================================

        dc_new = dc * mask

        # Forget gate
        df = dc_new * c_prev
        da_f = df * f_t * (1 - f_t)

        # Input gate
        di = dc_new * c_tilde
        da_i = di * i_t * (1 - i_t)

        # Candidate cell state
        dcandidate = dc_new * i_t
        da_c = dcandidate * (1 - c_tilde**2)

        # =====================================
        # 5. PARAMETER GRADIENTS
        # =====================================

        # Input gate
        grads["W_i"] += da_i.T @ x_t
        grads["U_i"] += da_i.T @ h_prev
        grads["b_i"] += np.sum(da_i, axis=0)

        # Forget gate
        grads["W_f"] += da_f.T @ x_t
        grads["U_f"] += da_f.T @ h_prev
        grads["b_f"] += np.sum(da_f, axis=0)

        # Output gate
        grads["W_o"] += da_o.T @ x_t
        grads["U_o"] += da_o.T @ h_prev
        grads["b_o"] += np.sum(da_o, axis=0)

        # Candidate cell state
        grads["W_c"] += da_c.T @ x_t
        grads["U_c"] += da_c.T @ h_prev
        grads["b_c"] += np.sum(da_c, axis=0)

        # =====================================
        # 6. EMBEDDING GRADIENTS
        # =====================================

        dx = (
            da_i @ parameters["W_i"] +
            da_f @ parameters["W_f"] +
            da_o @ parameters["W_o"] +
            da_c @ parameters["W_c"]
        )

        # Accumulate gradients for repeated token IDs
        np.add.at(
            grads["embedding_matrix"],
            token_ids,
            dx
        )

        # =====================================
        # 7. PREVIOUS STATE GRADIENTS
        # =====================================

        dh_prev = (
            da_i @ parameters["U_i"] +
            da_f @ parameters["U_f"] +
            da_o @ parameters["U_o"] +
            da_c @ parameters["U_c"]
        )

        dc_prev = dc_new * f_t

        # Propagate gradients through padding positions
        dh_next = dh_prev + dh_future * (1 - mask)

        dc_next = dc_prev + dc * (1 - mask)

    # Padding embedding is not trainable
    grads["embedding_matrix"][0] = 0

    return grads


In [146]:

def clip_gradients(grads, max_norm=5.0):

    # Calculate global gradient norm
    total_norm = np.sqrt(
        sum(np.sum(grad ** 2) for grad in grads.values())
    )

    # Calculate clipping scale
    scale = min(1.0, max_norm / (total_norm + 1e-8))

    # Clip all gradients
    clipped_grads = {
        name: grad * scale
        for name, grad in grads.items()
    }

    return clipped_grads, total_norm


In [147]:
clipped_grads, total_norm = clip_gradients(grads)

print("Original gradient norm:", total_norm)

clipped_norm = np.sqrt(sum(np.sum(grad ** 2) for grad in clipped_grads.values()))

print("Clipped gradient norm:", clipped_norm)

Original gradient norm: 0.1025419239133235
Clipped gradient norm: 0.1025419239133235


In [148]:
learning_rate=0.01
def update_parameters(parameters, clipped_grads, learning_rate):

    updated_parameters = {}

    # Update all trainable parameters
    for name, param in parameters.items():

        updated_parameters[name] = (param - learning_rate * clipped_grads[name])

    # Keep padding embedding fixed at zero
    updated_parameters["embedding_matrix"][0] = 0

    return updated_parameters


In [149]:
parameters = {
    "W_i": W_i, "U_i": U_i, "b_i": b_i,
    "W_f": W_f, "U_f": U_f, "b_f": b_f,
    "W_o": W_o, "U_o": U_o, "b_o": b_o,
    "W_c": W_c, "U_c": U_c, "b_c": b_c,
    "W_y": W_y, "b_y": b_y,
    "embedding_matrix": embedding_matrix
}

In [150]:
parameters = update_parameters(parameters,clipped_grads,learning_rate)

In [151]:
X_tokens_batch = X_train_padded[:batch_size]

probabilities, cache, h_final = forward_pass(
    X_tokens_batch,
    parameters
)

print("Probabilities shape:", probabilities.shape)
print("Cache length:", len(cache))
print("Final hidden state shape:", h_final.shape)
print("First review probabilities:", probabilities[0])
print("Probability sum:", probabilities[0].sum())

Probabilities shape: (32, 2)
Cache length: 100
Final hidden state shape: (32, 64)
First review probabilities: [0.4902596 0.5097404]
Probability sum: 1.0


In [152]:
grads = backward_pass(
    probabilities,
    y_batch,
    h_final,
    cache,
    parameters
)

for name, grad in grads.items():
    print(name, grad.shape)

W_i (64, 32)
U_i (64, 64)
b_i (64,)
W_f (64, 32)
U_f (64, 64)
b_f (64,)
W_o (64, 32)
U_o (64, 64)
b_o (64,)
W_c (64, 32)
U_c (64, 64)
b_c (64,)
W_y (2, 64)
b_y (2,)
embedding_matrix (10000, 32)


In [157]:
y_train_array = y_train.to_numpy()
y_val_array = y_val.to_numpy()

In [161]:

import time

def train_model(parameters, epochs, batch_size, learning_rate):

    history = {
        "train_loss": [],
        "val_loss": [],
        "val_accuracy": [],
        "epoch_time": []
    }

    best_val_loss = float("inf")
    best_epoch = 0

    num_train = len(X_train_padded)
    num_val = len(X_val_padded)

    total_start_time = time.time()

    for epoch in range(epochs):

        epoch_start_time = time.time()

        # =================================
        # 1. TRAINING
        # =================================

        indices = np.random.permutation(num_train)

        epoch_train_loss = 0

        for start in range(0, num_train, batch_size):

            batch_indices = indices[start:start + batch_size]

            X_batch = X_train_padded[batch_indices]
            y_batch = y_train_array[batch_indices]

            # Forward pass
            probabilities, cache, h_final = forward_pass(
                X_batch, parameters
            )

            # Loss calculation
            loss = cross_entropy_loss(probabilities, y_batch)

            # Backward pass
            grads = backward_pass(
                probabilities, y_batch, h_final, cache, parameters
            )

            # Gradient clipping
            clipped_grads, _ = clip_gradients(grads)

            # Parameter update
            parameters = update_parameters(
                parameters, clipped_grads, learning_rate
            )

            epoch_train_loss += loss * len(batch_indices)

        epoch_train_loss /= num_train

        # =================================
        # 2. VALIDATION
        # =================================

        val_loss = 0
        correct_predictions = 0

        for start in range(0, num_val, batch_size):

            end = start + batch_size

            X_batch = X_val_padded[start:end]
            y_batch = y_val_array[start:end]

            probabilities, _, _ = forward_pass(
                X_batch, parameters
            )

            batch_loss = cross_entropy_loss(
                probabilities, y_batch
            )

            val_loss += batch_loss * len(y_batch)

            predictions = np.argmax(probabilities, axis=1)

            correct_predictions += np.sum(
                predictions == y_batch
            )

        val_loss /= num_val
        val_accuracy = correct_predictions / num_val

        # =================================
        # 3. SAVE BEST MODEL
        # =================================

        if val_loss < best_val_loss:

            best_val_loss = val_loss
            best_epoch = epoch + 1

            np.savez_compressed(
                "best_lstm_model.npz",
                **parameters
            )

        # =================================
        # 4. EPOCH TIME
        # =================================

        epoch_elapsed = time.time() - epoch_start_time

        # =================================
        # 5. STORE RESULTS
        # =================================

        history["train_loss"].append(epoch_train_loss)
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(val_accuracy)
        history["epoch_time"].append(epoch_elapsed)

        print(
            f"Epoch {epoch + 1}/{epochs} | "
            f"Train Loss: {epoch_train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Accuracy: {val_accuracy:.4f} | "
            f"Time: {epoch_elapsed:.2f} sec"
        )

    total_elapsed = time.time() - total_start_time

    print("\nTraining completed.")
    print("Best epoch:", best_epoch)
    print("Best validation loss:", best_val_loss)
    print(f"Total training time: {total_elapsed / 60:.2f} minutes")

    return parameters, history


In [162]:
epochs = 5
batch_size = 32
learning_rate = 0.001

In [163]:
parameters, history = train_model(
    parameters,
    epochs,
    batch_size,
    learning_rate
)

Epoch 1/5 | Train Loss: 0.6933 | Val Loss: 0.6931 | Val Accuracy: 0.5024 | Time: 243.24 sec
Epoch 2/5 | Train Loss: 0.6933 | Val Loss: 0.6931 | Val Accuracy: 0.5007 | Time: 229.95 sec
Epoch 3/5 | Train Loss: 0.6933 | Val Loss: 0.6931 | Val Accuracy: 0.5016 | Time: 237.04 sec
Epoch 4/5 | Train Loss: 0.6933 | Val Loss: 0.6931 | Val Accuracy: 0.4997 | Time: 224.77 sec
Epoch 5/5 | Train Loss: 0.6933 | Val Loss: 0.6931 | Val Accuracy: 0.5008 | Time: 231.74 sec

Training completed.
Best epoch: 5
Best validation loss: 0.6930947709028108
Total training time: 19.45 minutes
